# 03 · Skewers

`SkewerBundle` places the sampling points of N_sk skewers, sets the photon directions and solves a(x) along each one
with a `Cosmology` (02). Its points and line elements are where every field is sampled (04) and the dual segments
where the simulation gas is deposited (05). Runtime: ~15 s.

**Requires:** a cosmology (02) to solve a(x), or `manual_scale_factors`.  
**Independent of:** lines, fields.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, use_backend, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM,
                                      CMAPS, LABELS, FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # dark style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

In [ ]:
plot_dag(('skewers',)); plt.show()

In [ ]:
from skewerjax.core import Cosmology, SkewerBundle, phys, path_geometry, plane_points, grid_skewer_points

## 1 · SkewerBundle: points, photon direction and a(x)

Each skewer has its initial condition (a0 at comoving distance x0) at its first (`ICs_defined_at_index=0`) or last
(`-1`) point, and that point is either where the light is emitted or where it is absorbed
(`ICs_at_emission_or_absorption`). These two choices fix the photon direction relative to the order of the points
(`array_direction`, ±1) and how a(x) is integrated.

In [ ]:
L = 1000.0
px = L * jnp.array([[0.0, 0.5, 0.5], [0.2, 0.5, 0.5], [0.3, 0.5, 0.5], [0.5, 0.5, 0.5], [1.0, 0.5, 0.5]])
planck = Cosmology(H0=phys.H0, Omega_r_0=9.1436e-5, Omega_m_0=0.3089, Omega_k_0=0.0, Omega_lambda_0=0.6911)
configs = [(0, "emission"), (0, "absorption"), (-1, "emission"), (-1, "absorption")]
sb = SkewerBundle(spatial_points=jnp.stack([px] * 4), a0=1.0, x0=0.0, ICs_defined_at_index=jnp.array([c[0] for c in configs]),
                  ICs_at_emission_or_absorption=[c[1] for c in configs], cosmologies=planck)
print("array_direction:", np.asarray(sb.array_direction), "| quadri_vectors (N_cosmo, N_sk, N_ppsk, 4):", sb.quadri_vectors.shape)
fig, ax = plt.subplots(figsize=(13, 6), layout="constrained")
for k, (i0, kind) in enumerate(configs):
    ax.plot(np.asarray(px[:, 0]), np.asarray(sb.scale_factors[0, k]), "o-", color=list(TOL.values())[k],
            label=f"ICs at point {i0} = {kind} (direction {int(sb.array_direction[k]):+d})")
ax.set(xlabel="x [Mpc]", ylabel="a(x)", title="The same points, four conventions: a = 1 where the ICs are set"); ax.legend(); plt.show()

## 2 · Geometry of a path

`path_geometry(points)` returns the path length s, the **line elements** w_i (half the distance to each neighbour, the
default quadrature weights of τ), the unit tangents and the midpoints m_{i±1/2} that bound the *dual segment* of every
point (where the simulation deposition of 05 puts the gas). Point layouts: `plane_points`, `grid_skewer_points`,
`axis_aligned_skewer_points`.

In [ ]:
from skewerjax.utils.plotting import plot_path_elements
zig = jnp.array([[0.0, 0.0, 0.0], [0.1, 0.2, 0.0], [0.35, 0.25, 0.0], [0.5, 0.6, 0.0], [0.9, 0.7, 0.0]])
g = path_geometry(zig)
fig, ax = plt.subplots(figsize=(15, 10), layout="constrained")
plot_path_elements(ax, zig)
ax.set_title("The geometry of a path: points, midpoints, dual segments, tangents and line elements")
plt.show()
print(f"sum of line elements {float(g['line_elements'].sum()):.6f} = path length {float(g['s'][-1]):.6f}")

*What each element is, and where it enters.*
- **Sampling points x_i** (white): where every field is evaluated (`Fields.sample`) and where a(x) is solved.
- **Midpoints m_{i±1/2}** (yellow diamonds): halfway to each neighbour. They bound the **dual segment** of point i,
  the stretch of path that point i stands for (alternating blue / cyan bands). The two end points own half a segment.
- **Line elements w_i** (boxes) = the length of the dual segment, ½(|x_{i+1} − x_i| + |x_i − x_{i−1}|). They are the
  quadrature weights of the optical depth, τ = 𝒞 Σ_i w_i n_i a_i⁻² φ_i, and they tile the path: Σ w_i is the path length.
- **Unit tangents t̂_i** (red): the direction of the path at each point (central difference). The line-of-sight
  peculiar velocity is v_∥ = v · t̂_i, and in a simulation the deposition (05) cuts every dual segment into the pieces
  of the cells it crosses.

### The paths in three dimensions

The same objects in three dimensions: a 3 × 3 grid of straight skewers along z (`grid_skewer_points`), an oblique straight
skewer and a helix, all in one 10 cMpc box and coloured by the scale factor that `SkewerBundle` solves along each
path (Planck 2018, the observer at the end of each path at z = 2). The colour changes along the line of sight only:
a(x) is a function of the comoving distance travelled by the light.
On the right, the first part of the helix with the elements of the figure above, every 10th point (so the line elements shown are those of the thinned path).

In [ ]:
from skewerjax.utils.plotting import plot_paths_3d, plot_path_elements_3d, dark_3d
Lb = 10.0
t = jnp.linspace(0.0, 1.0, 400)
grid9 = grid_skewer_points(Lb, 3, 400, direction="z").reshape(9, 400, 3).astype(jnp.float64)
oblique = jnp.stack([Lb * t, 0.15 * Lb + 0.7 * Lb * t, 0.9 * Lb - 0.8 * Lb * t], -1)
helix = jnp.stack([Lb / 2 + 0.35 * Lb * jnp.cos(3 * jnp.pi * t), Lb / 2 + 0.35 * Lb * jnp.sin(3 * jnp.pi * t), Lb * t], -1)
paths = jnp.concatenate([grid9, oblique[None], helix[None]])
sb3 = SkewerBundle(spatial_points=paths, a0=1 / 3, x0=0.0, ICs_defined_at_index=-1, ICs_at_emission_or_absorption="absorption",
                   cosmologies=planck)
a3 = np.asarray(sb3.scale_factors[0])
fig = plt.figure(figsize=(22, 10), layout="constrained")
ax = fig.add_subplot(1, 2, 1, projection="3d")
sm = plot_paths_3d(ax, paths, values=a3, box=Lb, cmap=CMAPS["sequential"], lw=2.5)
ax.view_init(22, -55)
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
plot_path_elements_3d(ax2, paths[-1, :121], every=10)                         # zoomed on the first part of the helix
dark_3d(ax2); ax2.set(xlabel="x [cMpc]", ylabel="y [cMpc]", zlabel="z [cMpc]")
ax2.view_init(28, -60)
ax2.legend(loc="upper left", fontsize=15)
ax.set_title("11 skewers in a 10 cMpc box, coloured by a(x)", color=FG, fontsize=18)
ax2.set_title("the first part of the helix (every 10th point): its elements", color=FG, fontsize=18)
fig.colorbar(sm, ax=[ax, ax2], shrink=0.6, label="scale factor a(x)")
plt.show()
g3 = path_geometry(paths)
print(f"path lengths: straight {float(g3['s'][0, -1]):.2f}, oblique {float(g3['s'][9, -1]):.2f}, helix {float(g3['s'][10, -1]):.2f} cMpc | "
      f"a from {a3.min():.5f} to {a3.max():.5f} | sum of line elements - path length: {float(jnp.max(jnp.abs(g3['line_elements'].sum(-1) - g3['s'][:, -1]))):.1e}")

**Options** of `SkewerBundle`: `a0`, `x0`, `ICs_defined_at_index` and `ICs_at_emission_or_absorption` (scalars or
per skewer), `los_directions` (else from the points), `manual_scale_factors` (else solved with `cosmologies`, which
may hold several backgrounds: one set of quadri-vectors per cosmology). The bundle is a pytree.

## 3 · Anatomy of a SkewerBundle

Everything a bundle stores, for one skewer (a helix sampled unevenly, so that the line elements vary). Every array is
a pytree leaf; `quadri_vectors` is the batch of events (a, x, y, z) that the field samplers (04) take.

In [ ]:
from skewerjax.utils.plotting import plot_skewer_anatomy
tu = jnp.linspace(0.0, 1.0, 120) ** 1.6                                     # uneven sampling along the path
helix_u = jnp.stack([Lb / 2 + 0.35 * Lb * jnp.cos(3 * jnp.pi * tu), Lb / 2 + 0.35 * Lb * jnp.sin(3 * jnp.pi * tu), Lb * tu], -1)[None]
sb_h = SkewerBundle(spatial_points=helix_u, a0=1 / 3, x0=0.0, ICs_defined_at_index=-1, ICs_at_emission_or_absorption="absorption",
                    cosmologies=planck)
fig, axs = plot_skewer_anatomy(sb_h, 0)
fig.suptitle("What a SkewerBundle stores (one skewer): positions, distances, line elements, a(x), photon directions", fontsize=21)
plt.show()